# sparse_release_failed_fold_interval_diagnostic_v1

Notebook-first failed-fold diagnostic for `sparse_release_bc_ppo_screen_v1`.

This notebook answers one narrow follow-up: why did the fold-local PPO handoff improve `wf_2022` relative return but widen MaxDD from the BC warm-start path?

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sparse_release_bc_ppo_screen_v1` / BC initialization from `sparse_release_bc_warmstart_screen_v1`.
- **Role**: `diagnose`.
- **Changed layer**: reusable path diagnostics only; this notebook regenerates the failed fold and does not promote an actor.
- **Target window**: `wf_2022` test (`2022-01-01` .. `2022-12-31`).
- **Accepted trade-off**: none for promotion; any MaxDD over 15% remains a blocker.
- **External reference**: Qlib/FinRL reproducible backtest discipline plus PBO/CSCV caution against selecting a PPO checkpoint from a validation uplift that fails the held-out fold.

External sources retained from the parent screen:

- Qlib: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Deep Momentum Networks: https://arxiv.org/abs/1904.04912
- Focal Loss / sparse-slice weighting caution: https://arxiv.org/abs/1708.02002
- Prioritized Experience Replay / sparse transition sampling: https://arxiv.org/abs/1511.05952
- Probability of Backtest Overfitting / CSCV: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import copy
import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.path_diagnostics import (
    compare_interval_feature_deltas,
    find_worst_drawdown_intervals,
    make_backtest_daily_frame,
)
from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'sparse_release_failed_fold_interval_diagnostic_v1'
SOURCE_EXPERIMENT = 'sparse_release_bc_ppo_screen_v1'
PROFILE_NAME = SOURCE_EXPERIMENT
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card' / SOURCE_EXPERIMENT
TARGET_CARD = ROOT / 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1'
TARGETS_PATH = TARGET_CARD / 'sparse_release_revised_targets.csv'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
CARD.mkdir(parents=True, exist_ok=True)

WF_2022 = {
    'fold': 'wf_2022',
    'target_fold': 'wf_2022_train',
    'train': ('2010-03-11', '2020-12-31'),
    'valid': ('2021-01-01', '2021-12-31'),
    'test': ('2022-01-01', '2022-12-31'),
}


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
profile = load_profile(PROFILE_NAME)
source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH)
train_end = pd.Timestamp(WF_2022['train'][1])
fold_targets = targets[
    (targets['fold'] == WF_2022['target_fold'])
    & (targets['datetime'] <= train_end)
].copy()
coverage = {
    'experiment': EXPERIMENT,
    'source_experiment': SOURCE_EXPERIMENT,
    'source_decision': source_summary.get('decision'),
    'profile': PROFILE_NAME,
    'fold': WF_2022['fold'],
    'target_fold': WF_2022['target_fold'],
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'target_rows': int(len(fold_targets)),
    'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
    'target_start': str(fold_targets['datetime'].min().date()) if len(fold_targets) else '',
    'target_end': str(fold_targets['datetime'].max().date()) if len(fold_targets) else '',
    'train': WF_2022['train'],
    'valid': WF_2022['valid'],
    'test': WF_2022['test'],
}
write_json(CARD / 'coverage.json', coverage)
pd.DataFrame([coverage]).to_csv(CARD / 'coverage.csv', index=False)
coverage


{'experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
 'source_experiment': 'sparse_release_bc_ppo_screen_v1',
 'source_decision': 'sparse_release_bc_ppo_screen_mixed_no_promotion',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'fold': 'wf_2022',
 'target_fold': 'wf_2022_train',
 'targets_path': 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1/sparse_release_revised_targets.csv',
 'feature_frame_path': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'target_rows': 253,
 'release_gate_rows': 40,
 'target_start': '2020-01-02',
 'target_end': '2020-12-31',
 'train': ('2010-03-11', '2020-12-31'),
 'valid': ('2021-01-01', '2021-12-31'),
 'test': ('2022-01-01', '2022-12-31')}

In [3]:
def run_backtest(actor, eval_df: pd.DataFrame, price_series: pd.Series, test_seg: tuple[str, str], label: str) -> tuple[dict, pd.DataFrame, dict]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        stress_threshold=profile.selection_score.stress_threshold,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    result = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
        trace_decisions=True,
    )
    daily = make_backtest_daily_frame(result, label=label, feature_frame=eval_df)
    metrics = {
        'return': float(result['return']),
        'sharpe': float(result['sharpe']),
        'max_dd': float(result['max_dd']),
        'buy_and_hold_return': float(split_metrics['buy_and_hold_return']),
        'relative_return': float(split_metrics['relative_return']),
        'avg_pos': float(split_metrics['avg_pos']),
        'stress_avg_pos': float(split_metrics['stress_avg_pos']),
        'trades': int(result['trades']),
        'decision_trace_rows': int(len(result.get('decision_trace', []))),
    }
    return result, daily, metrics


def gate(metrics: dict) -> bool:
    return bool(metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN and metrics.get('max_dd', 1e9) <= TARGET_MAX_DD)


In [4]:
init_qlib(QLIB_DATA_PATH)
set_seed(SEED)
train_config = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
obs_dim = observation_dim_for_profile(profile)

bc_fit = fit_state_objective_bc_actor_from_frame(
    frame,
    targets=fold_targets,
    profile=profile,
    train_config=train_config,
    sample_weight_config=None,
    sample_size=0,
)
bc_dataset = build_state_objective_bc_dataset_from_frame(
    frame,
    targets=fold_targets,
    profile=profile,
    sample_size=0,
)
bc_slice_metrics = evaluate_bc_actor_slice_fit(
    bc_fit.actor,
    bc_fit.obs,
    bc_fit.actions,
    bc_dataset.metadata,
    slice_cols=('release_gate_state', 'is_rebound_cluster', 'is_downtrend_state'),
)
bc_slice_summary = summarize_bc_slice_fit(
    bc_slice_metrics,
    required_slices=('release_gate_state',),
    min_accuracy=0.60,
    min_samples=20,
)
bc_slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)

# Rebuild only the failed fold, using the same fold-local train/valid/test split as the parent screen.
dataset = build_dataset(
    train_seg=WF_2022['train'],
    valid_seg=WF_2022['valid'],
    test_seg=WF_2022['test'],
    feature_packs=tuple(profile.feature.packs),
    profile_name=profile.name,
)
train_df = segment_frame(dataset, 'train')
valid_df = segment_frame(dataset, 'valid')
test_df = segment_frame(dataset, 'test')
price = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
test_price = price.loc[WF_2022['test'][0]:WF_2022['test'][1]]
valid_price = price.loc[WF_2022['valid'][0]:WF_2022['valid'][1]]

train_pickle = CARD / 'wf_2022_train_frame.pkl'
train_df.to_pickle(train_pickle)
initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)

policy, actor = build_policy(obs_dim, profile)
actor.load_state_dict(copy.deepcopy(bc_fit.actor.state_dict()))
actor.eval()
bc_test_result, bc_daily, bc_metrics = run_backtest(actor, test_df, test_price, WF_2022['test'], 'bc_initialization')
bc_valid_result, bc_valid_daily, bc_valid_metrics = run_backtest(actor, valid_df, valid_price, WF_2022['valid'], 'bc_initialization_valid')

actor.train()
selection = train_with_selection(
    policy=policy,
    actor=actor,
    initial_states=initial_states,
    obs_dim=obs_dim,
    profile=profile,
    valid_df=valid_df,
    smoke=False,
    test_df=None,
)
train_pickle.unlink(missing_ok=True)
actor.eval()
ppo_test_result, ppo_daily, ppo_metrics = run_backtest(actor, test_df, test_price, WF_2022['test'], 'ppo_selected')

history = pd.DataFrame(selection.get('history', []))
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_daily.to_csv(CARD / 'bc_daily_2022.csv', index=False)
ppo_daily.to_csv(CARD / 'ppo_selected_daily_2022.csv', index=False)

metrics = pd.DataFrame([
    {'path': 'bc_initialization', **bc_metrics, 'gate': gate(bc_metrics)},
    {'path': 'ppo_selected', **ppo_metrics, 'gate': gate(ppo_metrics)},
])
metrics['relative_delta_vs_bc'] = metrics['relative_return'] - float(bc_metrics['relative_return'])
metrics['max_dd_delta_vs_bc'] = metrics['max_dd'] - float(bc_metrics['max_dd'])
metrics.to_csv(CARD / 'path_metrics.csv', index=False)

display(metrics)
display(history)


[21614:MainThread](2026-05-26 03:03:54,207) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[21614:MainThread](2026-05-26 03:03:54,840) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[21614:MainThread](2026-05-26 03:03:54,843) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[21614:MainThread](2026-05-26 03:04:04,140) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,path,return,sharpe,max_dd,buy_and_hold_return,relative_return,avg_pos,stress_avg_pos,trades,decision_trace_rows,gate,relative_delta_vs_bc,max_dd_delta_vs_bc
0,bc_initialization,-16.534504,-0.804593,18.396429,-76.614257,36.419124,0.280667,0.206395,25,188,False,0.000000,0.000000
1,ppo_selected,-33.242355,-0.898307,38.682694,-76.614257,41.770333,0.318036,0.240116,29,188,False,5.351209,20.286264


,iter,score,return,sharpe,max_dd,avg_pos,stress_avg_pos,stress_days,buy_and_hold_return,relative_return
0,0,40.862227,20.302932,1.101080,12.185863,0.330955,0.320833,3.0,78.905912,-58.602980
1,4,59.824124,35.355293,1.318799,15.892958,0.463466,0.442889,3.0,78.905912,-43.550619


In [5]:
ppo_intervals = find_worst_drawdown_intervals(ppo_daily, top_n=5, min_depth=0.05)
bc_intervals = find_worst_drawdown_intervals(bc_daily, top_n=5, min_depth=0.05)
ppo_intervals.to_csv(CARD / 'ppo_worst_drawdown_intervals.csv', index=False)
bc_intervals.to_csv(CARD / 'bc_worst_drawdown_intervals.csv', index=False)

feature_cols = [
    col for col in ppo_daily.select_dtypes(include=[np.number]).columns
    if col not in {'account', 'nav', 'daily_return', 'peak', 'drawdown', 'position'}
]
feature_deltas = compare_interval_feature_deltas(
    candidate_daily=ppo_daily,
    baseline_daily=bc_daily,
    intervals=ppo_intervals,
    feature_cols=feature_cols,
)
feature_deltas_top = feature_deltas.groupby('interval_rank', group_keys=False).head(15).reset_index(drop=True) if not feature_deltas.empty else feature_deltas
feature_deltas_top.to_csv(CARD / 'feature_delta_by_interval.csv', index=False)

path_comparison = ppo_daily[['datetime', 'account', 'nav', 'daily_return', 'drawdown', 'position']].copy()
path_comparison = path_comparison.rename(columns={
    'account': 'ppo_account',
    'nav': 'ppo_nav',
    'daily_return': 'ppo_daily_return',
    'drawdown': 'ppo_drawdown',
    'position': 'ppo_position',
})
bc_path = bc_daily[['datetime', 'account', 'nav', 'daily_return', 'drawdown', 'position']].rename(columns={
    'account': 'bc_account',
    'nav': 'bc_nav',
    'daily_return': 'bc_daily_return',
    'drawdown': 'bc_drawdown',
    'position': 'bc_position',
})
path_comparison = path_comparison.merge(bc_path, on='datetime', how='left')
path_comparison['position_delta'] = path_comparison['ppo_position'] - path_comparison['bc_position']
path_comparison['account_delta_pct'] = (path_comparison['ppo_account'] / path_comparison['bc_account'] - 1.0) * 100.0
path_comparison['drawdown_delta'] = path_comparison['ppo_drawdown'] - path_comparison['bc_drawdown']
path_comparison.to_csv(CARD / 'path_comparison_2022.csv', index=False)

display(ppo_intervals)
display(feature_deltas_top.head(30))


,rank,peak_date,trough_date,start,end,rows,max_dd,max_dd_fraction,return,avg_pos,peak_nav,trough_nav
0,1,2022-03-30,2022-10-14,2022-03-30,2022-10-14,138,38.6827,0.386827,-38.6827,0.355923,1.015187,0.622486


,interval_rank,interval_start,interval_end,feature,interval_mean,reference_mean,delta,delta_z,interval_coverage,reference_coverage,candidate_avg_pos,baseline_avg_pos,position_delta,candidate_return,baseline_return,return_delta,abs_delta_z
0,1,2022-03-30,2022-10-14,below_ma63_days,0.658500,0.296627,0.361873,1.206386,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,1.206386
1,1,2022-03-30,2022-10-14,anchor_gap_180,-0.530836,-0.238172,-0.292664,-1.140639,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,1.140639
2,1,2022-03-30,2022-10-14,negative_gap_frequency_21,0.509662,0.420068,0.089594,0.992572,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.992572
3,1,2022-03-30,2022-10-14,below_anchor_ratio_63,0.981827,0.624150,0.357677,0.920724,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.920724
4,1,2022-03-30,2022-10-14,rolling_new_low_ratio_63,0.147343,0.099773,0.047570,0.824741,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.824741
5,1,2022-03-30,2022-10-14,drawdown_streak_63,1.000000,0.757511,0.242489,0.802471,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.802471
6,1,2022-03-30,2022-10-14,realized_vol_63,1.312249,1.200363,0.111885,0.729228,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.729228
7,1,2022-03-30,2022-10-14,trend_slope_63,-0.159887,-0.075359,-0.084528,-0.715153,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.715153
8,1,2022-03-30,2022-10-14,cum_drop_30,-0.339544,-0.248631,-0.090913,-0.640255,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.640255
9,1,2022-03-30,2022-10-14,drawdown_63,-0.485591,-0.376795,-0.108795,-0.628999,1.0,1.0,0.355923,0.180482,0.175441,-38.6827,-17.138,-21.5447,0.628999


In [6]:
top_interval = ppo_intervals.iloc[0].to_dict() if not ppo_intervals.empty else {}
if top_interval:
    start = pd.Timestamp(top_interval['peak_date'])
    end = pd.Timestamp(top_interval['trough_date'])
    in_top = (pd.to_datetime(path_comparison['datetime']) >= start) & (pd.to_datetime(path_comparison['datetime']) <= end)
    top_path = path_comparison.loc[in_top].copy()
else:
    top_path = pd.DataFrame()

position_delta_mean = float(top_path['position_delta'].mean()) if not top_path.empty else 0.0
position_delta_max = float(top_path['position_delta'].max()) if not top_path.empty else 0.0
account_delta_trough = float(top_path['account_delta_pct'].iloc[-1]) if not top_path.empty else 0.0
feature_readout = feature_deltas_top[feature_deltas_top['interval_rank'].eq(1)].head(8).to_dict(orient='records') if not feature_deltas_top.empty else []

ppo_vs_bc_relative_delta = float(ppo_metrics['relative_return'] - bc_metrics['relative_return'])
ppo_vs_bc_dd_delta = float(ppo_metrics['max_dd'] - bc_metrics['max_dd'])
failed_gate_reason = []
if ppo_metrics['relative_return'] < TARGET_RELATIVE_RETURN:
    failed_gate_reason.append('relative_return_below_target')
if ppo_metrics['max_dd'] > TARGET_MAX_DD:
    failed_gate_reason.append('maxdd_above_target')
if ppo_vs_bc_dd_delta > 2.0:
    failed_gate_reason.append('ppo_widened_drawdown_vs_bc')

conclusion = [
    f"Rebuilt wf_2022 failed fold: BC relative={bc_metrics['relative_return']:.2f}pp / MaxDD={bc_metrics['max_dd']:.2f}%; selected PPO relative={ppo_metrics['relative_return']:.2f}pp / MaxDD={ppo_metrics['max_dd']:.2f}%.",
    f"PPO gained {ppo_vs_bc_relative_delta:.2f}pp relative return but widened MaxDD by {ppo_vs_bc_dd_delta:.2f}pp, so validation-score selection is not sufficient for this handoff.",
]
if top_interval:
    conclusion.append(
        f"Worst selected-PPO interval is {top_interval['peak_date']} .. {top_interval['trough_date']} with MaxDD={top_interval['max_dd']:.2f}% and avg_pos={top_interval['avg_pos']:.3f}."
    )
    conclusion.append(
        f"Inside that interval, PPO held {position_delta_mean:.3f} average position more than BC (max delta {position_delta_max:.3f}); account at trough was {account_delta_trough:.2f}% below the BC path."
    )
if feature_readout:
    top_features = ', '.join(f"{row['feature']}({row['delta_z']:.2f}z)" for row in feature_readout[:5])
    conclusion.append(f"Top interval state differences versus the rest of 2022: {top_features}.")
conclusion.append(
    'Decision: do not lengthen PPO from this checkpoint. The next mechanism should be a state-conditioned drawdown controller/checkpoint objective or interval-level risk gate before any PPO handoff.'
)

selection_payload = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_experiment': SOURCE_EXPERIMENT,
    'decision': 'ppo_failed_wf_2022_drawdown_interval_no_longer_training',
    'promotion_allowed': False,
    'profile': PROFILE_NAME,
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'diagnostics'],
    'fold': WF_2022['fold'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'failed_gate_reason': failed_gate_reason,
    'next_mechanism': 'state_conditioned_drawdown_controller_or_checkpoint_objective',
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/1904.04912',
        'https://arxiv.org/abs/1708.02002',
        'https://arxiv.org/abs/1511.05952',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'path_metrics.csv',
        'bc_daily_2022.csv',
        'ppo_selected_daily_2022.csv',
        'path_comparison_2022.csv',
        'bc_worst_drawdown_intervals.csv',
        'ppo_worst_drawdown_intervals.csv',
        'feature_delta_by_interval.csv',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'source_experiment': SOURCE_EXPERIMENT,
    'decision': selection_payload['decision'],
    'coverage': coverage,
    'bc_metrics': bc_metrics,
    'ppo_selected_metrics': ppo_metrics,
    'bc_valid_metrics': bc_valid_metrics,
    'selection_history': history.to_dict(orient='records'),
    'bc_slice_summary': bc_slice_summary,
    'top_interval': top_interval,
    'top_interval_position_delta_mean': position_delta_mean,
    'top_interval_position_delta_max': position_delta_max,
    'top_interval_account_delta_trough_pct': account_delta_trough,
    'top_interval_feature_readout': feature_readout,
    'conclusion': conclusion,
    'selection': selection_payload,
}
write_json(CARD / 'selection.json', selection_payload)
write_json(CARD / 'summary.json', summary)
for line in conclusion:
    display(Markdown(f'- {line}'))
selection_payload


- Rebuilt wf_2022 failed fold: BC relative=36.42pp / MaxDD=18.40%; selected PPO relative=41.77pp / MaxDD=38.68%.

- PPO gained 5.35pp relative return but widened MaxDD by 20.29pp, so validation-score selection is not sufficient for this handoff.

- Worst selected-PPO interval is 2022-03-30 .. 2022-10-14 with MaxDD=38.68% and avg_pos=0.356.

- Inside that interval, PPO held 0.175 average position more than BC (max delta 0.316); account at trough was -25.16% below the BC path.

- Top interval state differences versus the rest of 2022: below_ma63_days(1.21z), anchor_gap_180(-1.14z), negative_gap_frequency_21(0.99z), below_anchor_ratio_63(0.92z), rolling_new_low_ratio_63(0.82z).

- Decision: do not lengthen PPO from this checkpoint. The next mechanism should be a state-conditioned drawdown controller/checkpoint objective or interval-level risk gate before any PPO handoff.

{'experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
 'created_at': '2026-05-25T19:06:07.896404+00:00',
 'source_experiment': 'sparse_release_bc_ppo_screen_v1',
 'decision': 'ppo_failed_wf_2022_drawdown_interval_no_longer_training',
 'promotion_allowed': False,
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'diagnostics'],
 'fold': 'wf_2022',
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'failed_gate_reason': ['maxdd_above_target', 'ppo_widened_drawdown_vs_bc'],
 'next_mechanism': 'state_conditioned_drawdown_controller_or_checkpoint_objective',
 'external_sources': ['https://github.com/microsoft/qlib',
  'https://arxiv.org/abs/2009.11189',
  'https://github.com/AI4Finance-Foundation/FinRL',
  'https://arxiv.org/abs/2011.09607',
  'https://arxiv.org/abs/1904.04912',
  'https://arxiv.org/abs/1708.02002',
  'https://arxiv.org/abs/1511.05952',
  'https://doi.org/10.21314/JCF.2016.322'],
 'sidecars'